# Tumor Size Analysis & Classification

Computes solid tumor volume in cm^3 for every BraTS 2023 GLI subject and
classifies each into one of three TNM-based size classes.

**Tumor definition:** solid tumor only (labels 1 + 3 = NCR + ET), excluding
peritumoral edema (label 2). BraTS 2023 GLI is 1mm isotropic, so
volume (cm^3) = voxel_count / 1000.

**Classification:** TNM T-stage diameter cutoffs converted to volume via the
sphere approximation V = (4/3)π(d/2)^3.

| Class | TNM | Diameter | Volume (sphere) |
|---|---|---|---|
| 0 = small  | T1 | ≤ 2 cm  | ≤ 4.19 cm^3 |
| 1 = medium | T2 | 2 – 5 cm | 4.19 – 65.45 cm^3 |
| 2 = large  | T3 | > 5 cm  | > 65.45 cm^3 |

Note: TNM staging is not officially applied to primary brain tumors (WHO CNS
uses histology/molecular markers). Adapted here for a reproducible size split.

**Outputs:** `tumor_size_labels.csv` (filename -> size_label), `tumor_size_summary.json`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


ORIG_DIR   = Path("/home/sven/Desktop/diffsuion/brain")                      
SEG_DIR    = Path("/home/sven/Desktop/diffsuion/Processed/brats/32_seg")      
OUTPUT_DIR = Path.cwd()                                                     


def sphere_volume(diameter_cm):
    # V = 4/3 * pi * (d/2)^3
    return (4 / 3) * np.pi * (diameter_cm / 2) ** 3


THRESHOLD_T1 = sphere_volume(2.0)   # small/medium boundary
THRESHOLD_T2 = sphere_volume(5.0)   # medium/large boundary

print(f"TNM volume thresholds:")
print(f"  small/medium: {THRESHOLD_T1:.2f} cm³ (d=2cm)")
print(f"  medium/large: {THRESHOLD_T2:.2f} cm³ (d=5cm)")


In [ ]:
subject_dirs = sorted(d for d in ORIG_DIR.iterdir() if d.is_dir())
sample_seg = next(subject_dirs[0].glob("*-seg.nii.gz"))
voxel_mm3 = float(np.prod(nib.load(str(sample_seg)).header.get_zooms()))
print(f"{len(subject_dirs)} subjects, voxel volume = {voxel_mm3:.1f} mm³")

records = []
for subj_dir in tqdm(subject_dirs, desc="computing tumor volumes"):
    seg_files = list(subj_dir.glob("*-seg.nii.gz"))
    if not seg_files:
        continue
    seg = nib.load(str(seg_files[0])).get_fdata()
    solid_voxels = int(((seg == 1) | (seg == 3)).sum())     # NCR + ET, excluding edema
    records.append({
        "subject":           subj_dir.name,
        "filename":          subj_dir.name + ".nii.gz",     
        "tumor_voxels_orig": solid_voxels,
        "volume_cm3":        solid_voxels * voxel_mm3 / 1000.0,
    })

df = pd.DataFrame(records)

seg_files_32 = {f.name for f in SEG_DIR.glob("*.nii.gz")}
df = df[df["filename"].isin(seg_files_32)].reset_index(drop=True)
print(f"matched to 32³ files: {len(df)}")


In [ ]:
print(f"\nvolume stats (solid tumor only, cm³):")
print(f"  min={df['volume_cm3'].min():.2f}  max={df['volume_cm3'].max():.2f}  "
      f"mean={df['volume_cm3'].mean():.2f}  median={df['volume_cm3'].median():.2f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df["volume_cm3"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(THRESHOLD_T1, color="green", linestyle="--", linewidth=2,
           label=f"T1/T2: {THRESHOLD_T1:.1f} cm³ (d=2cm)")
ax.axvline(THRESHOLD_T2, color="red",   linestyle="--", linewidth=2,
           label=f"T2/T3: {THRESHOLD_T2:.1f} cm³ (d=5cm)")
ax.set_xlabel("tumor volume (cm³)")
ax.set_ylabel("subjects")
ax.set_title("BraTS 2023 GLI tumor volume distribution with TNM-based thresholds")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
df["size_label"] = 1                                       # default: medium
df.loc[df["volume_cm3"] <= THRESHOLD_T1, "size_label"] = 0  # small
df.loc[df["volume_cm3"] >  THRESHOLD_T2, "size_label"] = 2  # large

names = {0: "small (T1)", 1: "medium (T2)", 2: "large (T3)"}
counts = df["size_label"].value_counts().sort_index().to_dict()

print("class distribution:")
for label, count in counts.items():
    subset = df[df["size_label"] == label]
    print(f"  {label} ({names[label]:12s}): {count:4d} ({count / len(df) * 100:.1f}%)"
          f"  range {subset['volume_cm3'].min():.1f}–{subset['volume_cm3'].max():.1f} cm³")


In [ ]:
examples = {}
for label in [0, 1, 2]:
    subset = df[df["size_label"] == label].sort_values("volume_cm3")
    if len(subset) > 0:
        examples[label] = subset.iloc[len(subset) // 2]

fig, axes = plt.subplots(len(examples), 3, figsize=(14, 5 * len(examples)))
if len(examples) == 1:
    axes = axes[np.newaxis, :]

for row, (label, info) in enumerate(examples.items()):
    seg_file = next((ORIG_DIR / info["subject"]).glob("*-seg.nii.gz"))
    vol = nib.load(str(seg_file)).get_fdata()
    solid = ((vol == 1) | (vol == 3)).astype(float)

    slices = [int(solid.sum(axis=tuple(j for j in range(3) if j != a)).argmax()) for a in range(3)]
    views = [vol[slices[0]], vol[:, slices[1]], vol[:, :, slices[2]]]
    titles = [f"axial {slices[0]}", f"coronal {slices[1]}", f"sagittal {slices[2]}"]

    for col, (img, title) in enumerate(zip(views, titles)):
        axes[row, col].imshow(img, cmap="nipy_spectral", vmin=0, vmax=3)
        axes[row, col].set_title(title)
        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(f"{names[label].upper()}\n{info['volume_cm3']:.1f} cm³",
                            fontsize=11, rotation=0, labelpad=70, va="center")

plt.suptitle("example segmentations by TNM size class (solid tumor only)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
csv_path  = OUTPUT_DIR / "tumor_size_labels.csv"
json_path = OUTPUT_DIR / "tumor_size_summary.json"

df[["filename", "volume_cm3", "size_label"]].to_csv(csv_path, index=False)

summary = {
    "classification":              "TNM T-stage (sphere volume approximation)",
    "formula":                     "V = (4/3) * pi * (d/2)^3",
    "tumor_definition":            "labels 1 (NCR) + 3 (ET), excluding label 2 (edema)",
    "threshold_T1_T2_cm3":         round(float(THRESHOLD_T1), 2),
    "threshold_T2_T3_cm3":         round(float(THRESHOLD_T2), 2),
    "threshold_T1_T2_diameter_cm": 2.0,
    "threshold_T2_T3_diameter_cm": 5.0,
    "label_names":                 {str(k): v for k, v in names.items()},
    "counts":                      {names[k]: int(v) for k, v in counts.items()},
    "total":                       len(df),
    "voxel_spacing_mm":            "1.0 x 1.0 x 1.0 (BraTS 2023 GLI, SRI24 template)",
    "volume_stats_cm3": {
        "min":    round(float(df["volume_cm3"].min()),    2),
        "max":    round(float(df["volume_cm3"].max()),    2),
        "mean":   round(float(df["volume_cm3"].mean()),   2),
        "median": round(float(df["volume_cm3"].median()), 2),
        "std":    round(float(df["volume_cm3"].std()),    2),
    },
    "note": ("TNM staging is not officially applied to primary brain tumors. "
             "Thresholds adapted from general TNM T-stage diameter cutoffs "
             "for demonstration purposes."),
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"saved {len(df)} rows to {csv_path}")
print(f"saved summary to {json_path}")
df[["filename", "volume_cm3", "size_label"]].head()
